# 04 · Corpus textual (corregido) — Memoria Chilena

**Jupyter local:** ejecutar todas las celdas. No se necesita repetir las etapas 1–3 ni mover ningún ZIP. La ruta se resuelve desde `notebooks/`.

**Qué corrige esta versión**

- Descarga por streaming al disco (no mantiene PDF enormes en memoria), con límite de seguridad de 350 MB por archivo.
- Reutiliza PDF locales válidos, así que no los descarga otra vez.
- Genera automáticamente `data/output/etapa4.zip` y una muestra para revisar la calidad textual.
- Separa corpus principal y registros RDF contextuales sin convertir vínculos curatoriales en hechos históricos.

**Alcance:** los seis PDF rotulados «Patrimonio cultural común» son candidatos para trabajo local; la etiqueta no concede por sí sola derechos de redistribución. Los cuatro restantes no se descargan. **No se hace OCR ni se autoriza indexar ciegamente textos defectuosos.** Los números de página son los del PDF, no necesariamente los impresos. Los PDF no se incluyen en el ZIP de resultados ni deben subirse al GitHub.

In [1]:
from pathlib import Path
import io, json, re, time, hashlib, zipfile, csv, sys, unicodedata
from urllib.parse import urlparse
from datetime import datetime, timezone
import pandas as pd
import requests
from rdflib import Graph, URIRef, BNode
import fitz

# Si el notebook está en notebooks/, ROOT es su carpeta padre.
CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if not (ROOT / "data").exists():
    raise FileNotFoundError(f"No encuentro la carpeta data/ desde {CWD}")
INPUT = ROOT / "data/output/MEMORIA_CHILENA_RESULTADOS.zip"
if not INPUT.is_file():
    INPUT = ROOT / "data/input/MEMORIA_CHILENA_RESULTADOS.zip"
OUTPUT = ROOT / "data/output/etapa4"
PDF_DIR = ROOT / "data/pdf"
OUTPUT.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT.is_file(), f"No encuentro {INPUT}"
print('Proyecto:', ROOT, '\nArchivo de etapas previas:', INPUT)

Proyecto: C:\Users\jvera\Documents\memoria_chilena 
Archivo de etapas previas: C:\Users\jvera\Documents\memoria_chilena\data\input\MEMORIA_CHILENA_RESULTADOS.zip


C:\Users\jvera\anaconda3\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [2]:
# Descomprimir en memoria, sin copiar ni mover archivos
with zipfile.ZipFile(INPUT) as total:
    assert 'etapa3.zip' in total.namelist(), 'Falta etapa3.zip'
    with zipfile.ZipFile(io.BytesIO(total.read('etapa3.zip'))) as etapa3:
        audit = pd.read_csv(io.BytesIO(etapa3.read('auditoria_documentos.csv'))).fillna('')
        downloads = pd.read_csv(io.BytesIO(etapa3.read('enlaces_descarga.csv'))).fillna('')
        graph_bytes = etapa3.read('grafo_ampliado.ttl')
        stage3_report = json.loads(etapa3.read('informe_etapa3.json').decode('utf-8'))

assert len(audit) == 10, f"Esperábamos 10 fichas: hay {len(audit)}"
assert audit['url'].is_unique, 'URLs duplicadas en auditoría'
print('Documentos:', len(audit), ' | Enlaces PDF:', len(downloads))
print(audit[['titulo_rdf','derechos_declarados']].to_string(index=False))

Documentos: 10  | Enlaces PDF: 10
                                                                                                                                                                                                                                                                         titulo_rdf       derechos_declarados
                                                                                                                                                                                                                                                              La Tirana de Tarapacá                          
                                                                                                                                                                                                                                                              El Altiplano de Arica                          
                                                            

In [3]:
# Clasificación explícita: ninguna fuente con derechos desconocidos/protegidos se descarga
rights_ok = 'Patrimonio cultural común'
audit['descarga_permitida_por_politica'] = audit['derechos_declarados'].eq(rights_ok)
links = downloads[['pagina','url_declarada']].drop_duplicates('pagina')
items = audit.merge(links, left_on='url', right_on='pagina', how='left')
items['motivo'] = items['descarga_permitida_por_politica'].map({True: 'candidato_catalogo_comun', False: 'sin_autorizacion_automatica'})
items['url_declarada'] = items['url_declarada'].fillna('')
items.to_csv(OUTPUT/'inventario_derechos.csv',index=False,encoding='utf-8-sig')
print('Candidatos:', int(items['descarga_permitida_por_politica'].sum()), '| Bloqueados:', int((~items['descarga_permitida_por_politica']).sum()))

Candidatos: 6 | Bloqueados: 4


In [4]:
# Descarga local por streaming: el límite anterior de 35 MB omitía cuatro PDF.
# Conserva PDFs ya descargados en data/pdf y genera registro detallado.
DESCARGAR = True
PAUSA_SEGUNDOS = 2.5
MAX_MB_PDF = 350          # Límite de seguridad; no se mantienen los PDF en memoria
TIMEOUT = (20, 180)       # (conexión, espera de lectura) por petición
DOMINIOS = {'www.memoriachilena.gob.cl', 'memoriachilena.gob.cl'}
headers = {'User-Agent': 'MemoriaChilenaResearchPilot/0.2 (investigacion academica; ver README en repositorio)'}
session = requests.Session()
session.headers.update(headers)

def hash_archivo(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for fragmento in iter(lambda: f.read(1024*1024), b''):
            h.update(fragmento)
    return h.hexdigest()

def validar_pdf(path):
    with open(path, 'rb') as f:
        if f.read(5) != b'%PDF-':
            raise ValueError('No contiene una cabecera PDF')
    with fitz.open(path) as d:
        if len(d) < 1: raise ValueError('PDF sin páginas')
        return len(d)

logs = []
max_bytes = MAX_MB_PDF * 1024 * 1024
for _, r in items.iterrows():
    url = str(r['url_declarada']).strip()
    mcid = str(r['mcid_rdf']).strip()
    rec = {'mcid': mcid, 'ficha': r['url'], 'pdf_url': url,
           'derechos': r['derechos_declarados'], 'estado': '',
           'bytes': 0, 'sha256': '', 'paginas': '', 'detalle': ''}
    path = PDF_DIR / f'{mcid}.pdf'
    tmp = PDF_DIR / f'{mcid}.pdf.part'
    if not r['descarga_permitida_por_politica']:
        rec['estado'] = 'omitido_derechos'; logs.append(rec); continue
    if not url or urlparse(url).hostname not in DOMINIOS:
        rec['estado'] = 'omitido_url_no_verificada'; logs.append(rec); continue

    if path.is_file():
        try:
            n = validar_pdf(path)
            rec.update(estado='cache_existente', bytes=path.stat().st_size,
                       sha256=hash_archivo(path), paginas=n)
            logs.append(rec)
            print(mcid, rec['estado'], f'({n} páginas)')
            continue
        except Exception as e:
            rec['detalle'] = 'Caché local inválida: ' + str(e)[:120]
            path.unlink(missing_ok=True)

    if not DESCARGAR:
        rec['estado'] = 'sin_red_configurada'; logs.append(rec); continue

    hubo_consulta = False
    try:
        url_https = url.replace('http://', 'https://', 1)
        hubo_consulta = True
        with session.get(url_https, timeout=TIMEOUT, stream=True, allow_redirects=True) as response:
            response.raise_for_status()
            if urlparse(response.url).hostname not in DOMINIOS:
                raise ValueError('Redirección fuera del dominio autorizado')
            declarado = response.headers.get('Content-Length', '').strip()
            if declarado.isdecimal() and int(declarado) > max_bytes:
                raise ValueError(f'PDF excede el límite de {MAX_MB_PDF} MB (Content-Length)')
            h = hashlib.sha256()
            total = 0
            with open(tmp, 'wb') as f:
                for bloque in response.iter_content(chunk_size=1024*1024):
                    if not bloque: continue
                    total += len(bloque)
                    if total > max_bytes:
                        raise ValueError(f'PDF excede el límite de {MAX_MB_PDF} MB')
                    f.write(bloque)
                    h.update(bloque)
        n = validar_pdf(tmp)
        tmp.replace(path)
        rec.update(estado='pdf_ok', bytes=total, sha256=h.hexdigest(), paginas=n)
    except Exception as e:
        mensaje = str(e)
        rec['estado'] = 'excede_limite' if 'excede el límite' in mensaje else 'error_descarga'
        rec['detalle'] = mensaje[:250]
    finally:
        tmp.unlink(missing_ok=True)
        if hubo_consulta: time.sleep(PAUSA_SEGUNDOS)
    logs.append(rec)
    print(mcid, rec['estado'], f"({rec['bytes']/1048576:.1f} MB)" if rec['bytes'] else rec['detalle'])

log_df = pd.DataFrame(logs)
log_df.to_csv(OUTPUT/'registro_descargas.csv', index=False, encoding='utf-8-sig')
print('\nEstado de descarga:')
print(log_df['estado'].value_counts().to_string())

MC0014753 cache_existente (181 páginas)
MC0014754 pdf_ok (104.2 MB)
MC0014755 pdf_ok (163.3 MB)
MC0014757 pdf_ok (111.8 MB)
MC0014760 cache_existente (157 páginas)
MC0014761 pdf_ok (162.1 MB)

Estado de descarga:
estado
omitido_derechos    4
pdf_ok              4
cache_existente     2


In [5]:
# Extracción de texto por página, con indicación de cobertura. No se ejecuta OCR.
pages=[]
for r in logs:
    if r['estado'] not in ('pdf_ok','cache_existente'): continue
    path = PDF_DIR/f"{r['mcid']}.pdf"
    try:
        with fitz.open(path) as doc:
            for n,page in enumerate(doc,1):
                txt=page.get_text('text',sort=True)
                txt=re.sub(r'[ \t]+',' ',txt).strip()
                pages.append({'mcid':r['mcid'],'pagina_pdf':n,'texto':txt,
                    'caracteres':len(txt),'texto_extraible':len(txt)>=80,
                    'fuente_pdf':r['pdf_url'],'ficha':r['ficha'], 'sha256_pdf':r['sha256']})
    except Exception as e:
        print('Error de extracción:',r['mcid'],str(e)[:160])
pages_df=pd.DataFrame(pages, columns=['mcid','pagina_pdf','texto','caracteres','texto_extraible','fuente_pdf','ficha','sha256_pdf'])
pages_df.to_csv(OUTPUT/'paginas_texto.csv',index=False,encoding='utf-8-sig')
if len(pages_df):
    print('Páginas:',len(pages_df),'| Páginas con texto >=80 caracteres:',int(pages_df['texto_extraible'].sum()))
else: print('No se extrajo texto; revisar registro_descargas.csv')

Páginas: 2641 | Páginas con texto >=80 caracteres: 1726


In [6]:
# Diagnóstico preliminar de cobertura y de legibilidad.
# IMPORTANTE: >=80 caracteres indica longitud, NO que el reconocimiento sea correcto.
qc_rows=[]
for _,p in pages_df.iterrows():
    txt = str(p['texto']) if pd.notna(p['texto']) else ''
    letras = sum(ch.isalpha() for ch in txt)
    alnum = sum(ch.isalnum() for ch in txt)
    simbolos = sum((not ch.isalnum()) and (not ch.isspace()) for ch in txt)
    tokens = re.findall(r'[^\W\d_]+', txt, flags=re.UNICODE)
    tokens_cortos = sum(len(t) <= 2 for t in tokens)
    qc_rows.append({'mcid':p['mcid'], 'pagina_pdf':int(p['pagina_pdf']),
       'caracteres':len(txt), 'supera_80_caracteres':len(txt)>=80,
       'proporcion_simbolos':round(simbolos/max(1,len(txt)),3),
       'proporcion_tokens_cortos':round(tokens_cortos/max(1,len(tokens)),3),
       'revision_visual_necesaria':True})
qc_df=pd.DataFrame(qc_rows, columns=['mcid','pagina_pdf','caracteres','supera_80_caracteres','proporcion_simbolos','proporcion_tokens_cortos','revision_visual_necesaria'])
qc_df.to_csv(OUTPUT/'auditoria_paginas.csv',index=False,encoding='utf-8-sig')

# Muestra legible por página para revisión humana: primeras 5 páginas de longitud >=80 de cada PDF.
muestras=[]
if len(pages_df):
    for mcid, grupo in pages_df.groupby('mcid'):
        candidatas=grupo[grupo['caracteres']>=80].sort_values('pagina_pdf').head(5)
        for _, p in candidatas.iterrows():
            muestras.append({'mcid':mcid,'pagina_pdf':int(p['pagina_pdf']),
                             'caracteres':int(p['caracteres']),
                             'muestra_500_caracteres':str(p['texto'])[:500]})
pd.DataFrame(muestras, columns=['mcid','pagina_pdf','caracteres','muestra_500_caracteres']).to_csv(
     OUTPUT/'muestras_calidad_texto.csv',index=False,encoding='utf-8-sig')
if len(pages_df):
    print('Cobertura por documento (longitud; no es garantía de calidad):')
    print(pages_df.groupby('mcid').agg(paginas=('pagina_pdf','count'),
          paginas_80_caracteres=('texto_extraible','sum'),
          caracteres=('caracteres','sum')).to_string())
print('Se generaron muestras_calidad_texto.csv y auditoria_paginas.csv para revisar OCR.')

Cobertura por documento (longitud; no es garantía de calidad):
           paginas  paginas_80_caracteres  caracteres
mcid                                                 
MC0014753      181                     58       49551
MC0014754      287                     53       40722
MC0014755      545                    472     1254299
MC0014757      570                    494      886712
MC0014760      157                     77      140464
MC0014761      901                    572     1121508
Se generaron muestras_calidad_texto.csv y auditoria_paginas.csv para revisar OCR.


In [7]:
# Fragmentos PROVISIONALES (no validados): nunca atraviesan límites de página; conservar fuente, hash y página.
MAX_CHARS=1100
OVERLAP=170
chunks=[]
for _,p in pages_df.iterrows():
    if not bool(p['texto_extraible']): continue
    s=p['texto']
    start=0
    while start<len(s):
        end=min(len(s),start+MAX_CHARS)
        if end<len(s):
            pos=s.rfind(' ',start+int(MAX_CHARS*.65),end)
            if pos>start: end=pos
        fragment=s[start:end].strip()
        if fragment:
            cid=f"{p['mcid']}_p{int(p['pagina_pdf']):04d}_c{len(chunks):06d}"
            chunks.append({'chunk_id':cid,'mcid':p['mcid'],'pagina_pdf':int(p['pagina_pdf']),
                'texto':fragment,'ficha':p['ficha'],'fuente_pdf':p['fuente_pdf'],
                'sha256_pdf':p['sha256_pdf'],'char_inicio':start,'char_fin':end})
        if end==len(s): break
        start=max(start+1,end-OVERLAP)
chunks_df=pd.DataFrame(chunks,columns=['chunk_id','mcid','pagina_pdf','texto','ficha','fuente_pdf','sha256_pdf','char_inicio','char_fin'])
chunks_df.to_csv(OUTPUT/'fragmentos.csv',index=False,encoding='utf-8-sig')
print('Fragmentos:',len(chunks_df))

Fragmentos: 4286


In [8]:
# Separar grafo del corpus principal de los registros externos/contextuales.
full = Graph().parse(data=graph_bytes.decode('utf-8'),format='turtle')
core_uris={URIRef(u) for u in items['url'].tolist()}
# En algunos registros, las URI del RDF usan una forma distinta de la ficha HTML.
# Detectar URIs catalogadas únicamente mediante identificadores declarados para evitar inferencias falsas.
from rdflib.namespace import DC, DCTERMS, RDF
namespaces = list(full.namespaces())
core = Graph(); context = Graph()
for p,ns in namespaces: core.bind(p,ns); context.bind(p,ns)
# Separación conservadora: todas las tripletas vinculadas al sujeto se guardan en core;
# nodos anónimos alcanzables por objetos se incluyen sin expandir a URI secundarias.
from collections import deque
start_nodes=set(core_uris)
# Sujeto RDF posiblemente diferente de la ficha: localizar por mcid como literal.
for subj,pred,obj in full:
    if isinstance(obj, __import__('rdflib').term.Literal) and str(obj).strip() in set(items['mcid_rdf'].astype(str)):
        if isinstance(subj,URIRef): start_nodes.add(subj)
queue=deque(start_nodes); seen=set()
while queue:
    node=queue.popleft()
    if node in seen: continue
    seen.add(node)
    for s,p,o in full.triples((node,None,None)):
        core.add((s,p,o))
        if isinstance(o,BNode): queue.append(o)
for triple in full:
    if triple not in core: context.add(triple)
assert len(core)+len(context)==len(full)
core.serialize(destination=str(OUTPUT/'grafo_corpus_principal.ttl'),format='turtle')
context.serialize(destination=str(OUTPUT/'grafo_contexto.ttl'),format='turtle')
print('Grafo total:',len(full),'| Principal:',len(core),'| Contexto:',len(context))
print('Sujetos principales candidatos:',len(start_nodes))

Grafo total: 3473 | Principal: 831 | Contexto: 2642
Sujetos principales candidatos: 10


In [9]:
# Informe y ZIP de resultados (sin PDFs, para proteger derechos y mantener GitHub liviano).
reporte={
 'fecha_utc':datetime.now(timezone.utc).isoformat(),
 'documentos_catalogados':int(len(items)),
 'documentos_elegibles_por_etiqueta':int(items['descarga_permitida_por_politica'].sum()),
 'documentos_con_pdf_local':int(log_df['estado'].isin(['pdf_ok','cache_existente']).sum()),
 'documentos_omitidos_por_derechos':int((log_df['estado']=='omitido_derechos').sum()),
 'paginas_extraidas':int(len(pages_df)),
 'paginas_con_texto_extraible':int(pages_df['texto_extraible'].sum()) if len(pages_df) else 0,
 'fragmentos_extraidos':int(len(chunks_df)),
 'triples_corpus_principal':int(len(core)),
 'triples_contexto':int(len(context)),
 'triples_grafo_total':int(len(full)),
 'metodo':'PyMuPDF extraccion nativa, sin OCR, sin embeddings',
 'documentos_detenidos_por_tamano':int((log_df['estado']=='excede_limite').sum()),
 'paginas_con_texto_min_80_no_validado':int(pages_df['texto_extraible'].sum()) if len(pages_df) else 0,
 'limite_mb_por_pdf':MAX_MB_PDF,
 'advertencias':['Los PDFs se descargan solo localmente y no se incluyen en ZIP/GitHub',
                'La etiqueta Patrimonio cultural común no garantiza derechos de redistribución',
                'Los PDF de obra protegida y de derechos no identificados no se descargan',
                'Los números de página corresponden al visor PDF, no necesariamente a la paginación impresa',
                'Los registros RDF relacionados no son hechos históricos confirmados',
                '>=80 caracteres no demuestra legibilidad: auditar muestras antes de crear embeddings']}
(OUTPUT/'informe_etapa4.json').write_text(json.dumps(reporte,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=ROOT/'data/output/etapa4.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.glob('*')): z.write(p,arcname=p.name)
print(json.dumps(reporte,ensure_ascii=False,indent=2))
print('\nZIP generado:',zip_path)

{
  "fecha_utc": "2026-11-09T18:27:47.933065+00:00",
  "documentos_catalogados": 10,
  "documentos_elegibles_por_etiqueta": 6,
  "documentos_con_pdf_local": 6,
  "documentos_omitidos_por_derechos": 4,
  "paginas_extraidas": 2641,
  "paginas_con_texto_extraible": 1726,
  "fragmentos_extraidos": 4286,
  "triples_corpus_principal": 831,
  "triples_contexto": 2642,
  "triples_grafo_total": 3473,
  "metodo": "PyMuPDF extraccion nativa, sin OCR, sin embeddings",
  "documentos_detenidos_por_tamano": 0,
  "paginas_con_texto_min_80_no_validado": 1726,
  "limite_mb_por_pdf": 350,
  "advertencias": [
    "Los PDFs se descargan solo localmente y no se incluyen en ZIP/GitHub",
    "La etiqueta Patrimonio cultural común no garantiza derechos de redistribución",
    "Los PDF de obra protegida y de derechos no identificados no se descargan",
    "Los números de página corresponden al visor PDF, no necesariamente a la paginación impresa",
    "Los registros RDF relacionados no son hechos históricos con